In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install git+https://github.com/openai/whisper.git praat-parselmouth librosa langchain_groq

# Importing API keys

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")
GROQ_API_KEY_2 = user_secrets.get_secret("GROQ_API_KEY_2")
HF_TOKEN = user_secrets.get_secret("HUGGING_FACE_ACCESS_TOKEN")
LLM_MODEL = user_secrets.get_secret("LLM_MODEL")


# Prompt_Template

In [ ]:
from typing import Literal, List, Annotated
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

class ClassificationOutput(BaseModel):
    category: Literal["query", "preference", "both"] = Field(
        ..., description="The type of input: query, preference, or both."
    )
    genre_filter: List[str] = Field(
        default_factory=list,
        description="List of broad topics like food, comedy, movie, books, etc., only for 'query'. Empty otherwise."
    )
    preference_type_filter: List[Literal["like", "dislike", "prefer", "avoid", "neutral"]] = Field(
        default_factory=list,
        description="List of user preference types, like prefer, avoid, etc. Only for 'query'. Empty otherwise."
    )

class FeatureObject(BaseModel):
    """Represents a single implicit feature extracted from conversation."""
    Standard_fine_tuning_prompt: str = Field(
        ..., description="A system prompt followed by user and assistant messages that demonstrate the implicit feature"
    )

class FeaturesOutput(BaseModel):
    """Output parser for implicit features extraction."""
    features: Annotated[List[str], Field(default_factory=list)]

class JudgeResult(BaseModel):
    verdict: int  # 1 if correct, 0 if wrong
    feedback: str  # short feedback


judge_parser = PydanticOutputParser(pydantic_object=JudgeResult)
parser = PydanticOutputParser(pydantic_object=ClassificationOutput)

classification_prompt = PromptTemplate(
    template="""
You are an AI assistant that classifies user inputs. Your task is to:

1. Determine if the input is a:
   - "query": a question
   - "preference": a statement expressing a like/dislike/preference (explicit feature)
   - "both": question and preference both

2. If the input is a "query" or "both", also extract:
   - genre_filter: list of general categories relevant to the query (e.g., ["food", "movie", "comedy", "books"])
   - preference_type_filter: inferred preferences from the user, such as ["like", "prefer", "avoid", "dislike"]

Examples:

Input: "What are the best romantic movies to watch?"
Output: category="query", genre_filter=["movie", "romantic"], preference_type_filter=["prefer"]

Input: "I dislike horror books."
Output: category="preference", genre_filter=[], preference_type_filter=[]

Input: "Can you suggest some vegetarian recipes? I want to avoid meat."
Output: category="both", genre_filter=["food", "vegetarian"], preference_type_filter=["avoid"]

Examples of INCORRECT responses:
- "The input is a query about movies." (Not JSON)
- "Here's the classification: json" (Extra text around JSON)
- "```json\njson\n```" (Markdown formatting)


Now classify the following user input:

User Input: {user_input}

Respond in this format:
{format_instructions}
""",
    input_variables=["user_input"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

extract_preferences_prompt = PromptTemplate(
    input_variables=["user_prompt"],
    template="""
You are a smart assistant helping to personalize user interactions.
Your job is to analyze the given prompt and determine if it contains:

1. Explicit preferences (clearly stated preferences like "I like", "I prefer", etc., one or two word genres like "movie", "book", "music", etc., and specific items like "comedy", "action", "romance", etc.)

If explicit or implicit preferences exist, return them in JSON with this format:
{{
  "explicit": [
    {{
      "type": "prefer",
      "genre": "<genre>",
      "specific item": "<specific item> (if applicable)",
      "conditions": {{
        "attribute": "value"
      }},
      "statement": "<Natural language statement describing this explicit preference, to the point explicit preference>"
    }}
  ],
}}

Make sure:
- Your JSON is valid and parseable.
- For explicit preferences, always include a "statement" summarizing them in natural language.
- Do NOT hallucinate conditions that are not implied.
- Only return the raw JSON object without any extra formatting like Markdown, explanation, or triple backticks.

Now analyze the following prompt and return the JSON:
---
Prompt: "{user_prompt}"
"""
)



strip_preferences_prompt = PromptTemplate(
    input_variables=["user_input"],
    template="""
You are an assistant that processes user inputs. Your task is to extract or rewrite the input so that it contains **only the main question or query**, without any preference-related details.

- Remove all explicit or implicit preferences, such as likes, dislikes, desires, or constraints (e.g., "I like", "avoid", "prefer", "dislike", etc.).
- Keep only the **core question** the user is asking.
- Do NOT include any reasoning, explanation, or formatting — just return the cleaned question.

Examples:

Input: "Can you suggest some good Italian restaurants? I dislike crowded places."
Output: "Can you suggest some good Italian restaurants?"

Input: "What movies should I watch? I prefer comedy and avoid horror."
Output: "What movies should I watch?"

Now process the following input:

Input: {user_input}
Output:
"""
)


# ================================
# Implicit feature prompt template
# ================================

# Create parser for the features output
implicit_features_parser = PydanticOutputParser(pydantic_object=FeaturesOutput)

extract_implicit_features_prompt = PromptTemplate(
    input_variables=["chat_session"],
    template="""
You are an AI system that extracts implicit features from a given chat session between a user and an assistant. 

Implicit features are:
- Unspoken preferences (e.g., preference for concise answers, a casual tone, technical detail level, etc.)
- Tone/style preferences (e.g., prefers humor, prefers formality, etc.)
- Behavioral patterns (e.g., often asks for examples, avoids certain topics, etc.)

Instructions:
1. ONLY include features that are actually supported by the conversation. Do not hallucinate.
2. If there are no relevant implicit features, output exactly: none
3. For each detected implicit feature, create one JSON object in the following format:

[{{
  "Standard_fine_tuning_prompt": "<|system|> [Briefly describe the implicit preference, tone, or style detected from the conversation].\\n<|user|> [User message(s) combined if few implicit features]\\n<|assistant|> [Assistant message(s)]"
}},
{{
  "Standard_fine_tuning_prompt": "<|system|> [Briefly describe the implicit preference, tone, or style detected from the conversation].\\n<|user|> [User message(s) combined if few implicit features]\\n<|assistant|> [Assistant message(s)]"
}}]

4. Multiple implicit features → multiple JSON objects inside a array.
5. Do not add explanations, comments, or extra text — ONLY return either:
   - "none"
   - OR an array of JSON objects in the exact structure above.

Now process the following chat session:
{chat_session}
"""
)


judge_prompt = PromptTemplate.from_template(
    """
You are an evaluator LLM acting as a strict judge. 
You will be given:
- Query: {query}
- Ground Truth: {ground_truth}
- Preference: {preference}
- Response: {response}

Your evaluation rules:
1. Compare the Response with the Ground Truth and check if it aligns with the Preference.
2. If the Response is correct → return JSON with feedback "Correct." and verdict=1.
3. If the Response is wrong → return JSON with a short and clear feedback, then verdict=0.
4. If the Response is lengthy there should be no penalty on it.

Return ONLY a valid JSON object in this format:
{{
  "verdict": <0 or 1>,
  "feedback": "<short feedback>"
}}

{format_instructions}
""",
    partial_variables={"format_instructions": judge_parser.get_format_instructions()},
)

# LLM_Judge

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnableSequence

def evaluate_response(query: str, ground_truth: str, preference: str, response: str, llm: ChatGroq):
    """Evaluates the response using LLM as Judge and returns structured JSON."""

    # Set LLM temperature to 0
    llm = llm.bind(temperature=0)

    # Build chain: Prompt → LLM → Parser
    judge_chain: RunnableSequence = judge_prompt | llm | judge_parser

    # Invoke with inputs
    result = judge_chain.invoke({
        "query": query,
        "ground_truth": ground_truth,
        "preference": preference,
        "response": response
    })

    # Convert Pydantic model to dict (JSON-serializable)
    return result.dict()


# Example usage
if __name__ == "__main__":
    pass
    # dotenv.load_dotenv()
    # GROQ_API_KEY = os.getenv("GROQ_API_KEY")
    # LLM_MODEL = os.getenv("LLM_MODEL")

    # # Initialize Groq LLaMA-3 LLM
    # llm = ChatGroq(
    #     api_key=GROQ_API_KEY,
    #     model_name=LLM_MODEL,
    # )
    # output = evaluate_response(
    #     query="What is the capital of France?",
    #     ground_truth="Paris",
    #     preference="Answer must be concise and factual",
    #     response="The capital of France is Paris.",
    #     llm=llm
    # )
    # print(output)


# Adapter Management

In [ ]:
import os
import torch
import json
import copy
from torch.nn.functional import cosine_similarity
import logging
from typing import Dict, List, Optional
from peft import PeftModel, LoraConfig, get_peft_model, TaskType
import numpy as np
from datetime import datetime

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class AdapterManager:
    def __init__(self, base_model_name: str, base_model_dir: str = "./kaggle/working/models"):
        """
        Initialize adapter manager
        
        Args:
            base_model_name: Name of the base model (e.g., 'llama-3.2-1B-instruct')
            base_model_dir: Directory to store model adapters
        """
        self.base_model_name = base_model_name
        self.base_model_dir = base_model_dir
        self.users = {}  # Track user adapters
        
        # Create directory structure if it doesn't exist
        os.makedirs(self.base_model_dir, exist_ok=True)
        
        # Paths for global adapter
        self.global_adapter_dir = os.path.join(self.base_model_dir, "global_adapter")
        os.makedirs(self.global_adapter_dir, exist_ok=True)
        
        # Keep track of aggregation stats
        self.aggregation_history = []
        
        logger.info(f"AdapterManager initialized for base model: {base_model_name}")
    
    def get_user_adapter_paths(self, user_id: str) -> Dict[str, str]:
        """
        Get paths for a user's local adapter only
        
        Args:
            user_id: The user identifier
        
        Returns:
            Dict with paths for local adapter
        """
        user_dir = os.path.join(self.base_model_dir, f"user_{user_id}")
        os.makedirs(user_dir, exist_ok=True)
        local_adapter_dir = os.path.join(user_dir, "local_adapter")
        os.makedirs(local_adapter_dir, exist_ok=True)
        return {
            "local": local_adapter_dir
        }
    
    def initialize_user(self, user_id: str, base_model, peft_config: LoraConfig):
        adapter_paths = self.get_user_adapter_paths(user_id)
        if user_id not in self.users:
            self.users[user_id] = {
                "local_adapter_path": adapter_paths["local"],
                "last_update": datetime.now().isoformat()
            }

        # Local adapter
        local_adapter_exists = os.path.exists(os.path.join(adapter_paths["local"], "adapter_model.safetensors"))
        local_adapter_dir = adapter_paths["local"].replace("\\", "/")
        global_adapter_dir = self.global_adapter_dir.replace("\\", "/")

        if local_adapter_exists:
            logger.info(f"Loading existing local adapter for user {user_id}")
            local_model = PeftModel.from_pretrained(base_model, local_adapter_dir)
        else:
            logger.info(f"Initializing new local adapter for user {user_id}")
            local_model = get_peft_model(base_model, peft_config)
            local_model.save_pretrained(local_adapter_dir, safe_serialization=True, save_adapter=True)


        # Always load the global adapter from the shared directory
        if os.path.exists(os.path.join(self.global_adapter_dir, "adapter_model.safetensors")):
            logger.info(f"Loading global adapter from shared directory for user {user_id}")
            global_model = PeftModel.from_pretrained(base_model, global_adapter_dir)
        else:
            logger.info(f"Initializing new global adapter in shared directory")
            global_model = get_peft_model(base_model, peft_config)
            global_model.save_pretrained(global_adapter_dir,safe_serialization=True, save_adapter=True)

        # Set parameters to trainable
        for name, param in local_model.named_parameters():
            if "lora" in name:
                param.requires_grad = True
        for name, param in global_model.named_parameters():
            if "lora" in name:
                param.requires_grad = True

        trainable_params = sum(p.numel() for p in local_model.parameters() if p.requires_grad)
        print(f"Trainable local adapter parameters for user {user_id}: {trainable_params}")

        trainable_params = sum(p.numel() for p in global_model.parameters() if p.requires_grad)
        print(f"Trainable global adapter parameters: {trainable_params}")

        return local_model, global_model
    
    def _copy_adapter(self, src_path: str, dst_path: str):
        """Copy adapter from src to dst with error handling"""
        config_path = os.path.join(src_path, "adapter_config.json")
        model_path = os.path.join(src_path, "adapter_model.safetensors")

        if not (os.path.exists(config_path) and os.path.exists(model_path)):
            logger.error(f"Adapter files not found in {src_path}")
            return False

        # Check for non-empty file
        if os.path.getsize(model_path) < 100:  # Arbitrary small size, adjust as needed
            logger.error(f"Adapter file {model_path} is empty or corrupted.")
            return False

        try:
            with open(config_path, 'r') as f:
                adapter_config = json.load(f)

            with open(os.path.join(dst_path, "adapter_config.json"), 'w') as f:
                json.dump(adapter_config, f)

            state_dict = torch.load(model_path, map_location="cpu", weights_only=False)
            torch.save(state_dict, os.path.join(dst_path, "adapter_model.safetensors"))
            return True
        except Exception as e:
            logger.error(f"Error copying adapter: {e}")
            return False
    
    def federated_aggregation(self, base_model, peft_config: LoraConfig, participating_users: Optional[List[str]] = None):
        """
        Perform federated aggregation of adapters
        """
        if not self.users:
            logger.warning("No users registered. Skipping aggregation.")
            return base_model

        users_to_aggregate = participating_users if participating_users else list(self.users.keys())
        logger.info(f"Starting federated aggregation with {len(users_to_aggregate)} users")

        # Step 2: Aggregate all user locals into the main global adapter
        self._aggregate_global_adapters(users_to_aggregate, base_model,peft_config)

        # Load the updated global model and return
        global_model = PeftModel.from_pretrained(base_model, self.global_adapter_dir)

        # Record this aggregation
        self.aggregation_history.append({
            "timestamp": datetime.now().isoformat(),
            "participating_users": users_to_aggregate,
            "user_count": len(users_to_aggregate)
        })

        return global_model
    
    def _aggregate_global_adapters(self, user_ids: List[str], base_model, peft_config):
        if not user_ids:
            return

        try:
            global_model = PeftModel.from_pretrained(base_model, self.global_adapter_dir.replace("\\", "/"))
        except Exception:
            global_model = get_peft_model(base_model, peft_config)

        global_state = global_model.state_dict()
        user_adapters = []

        for user_id in user_ids:
            adapter_paths = self.get_user_adapter_paths(user_id)
            try:
                user_model = PeftModel.from_pretrained(base_model, adapter_paths["local"].replace("\\", "/"))
                user_adapters.append(user_model.state_dict())
                logger.info(f"Loaded user {user_id} local adapter")
            except Exception as e:
                logger.error(f"Failed to load local adapter for user {user_id}: {e}")

        if not user_adapters:
            logger.warning("No valid user adapters found. Skipping global aggregation.")
            return

        # Total user contribution = 40%, shared equally among all users
        N = len(user_adapters)
        total_user_weight = 0.4
        per_user_weight = total_user_weight / N
        global_weight = 1.0 - total_user_weight  # remaining 80%

        for key in global_state:
            if "lora" in key:
                agg = None
                # Sum each user's weights with equal share of 20%
                for user_state in user_adapters:
                    if key in user_state:
                        if agg is None:
                            agg = per_user_weight * user_state[key].clone()
                        else:
                            agg += per_user_weight * user_state[key]
                # Add 80% from the previous global
                if agg is not None and global_weight > 0:
                    agg += global_weight * global_state[key]
                if agg is not None:
                    global_state[key] = agg

        for name, param in global_model.named_parameters():
            if "lora" in name and name in global_state:
                param.data = global_state[name].to(param.device)

        global_model.save_pretrained(self.global_adapter_dir.replace("\\", "/"))
        logger.info(f"Global adapter updated: {len(user_adapters)} users share 20% total contribution, {global_weight*100:.1f}% from previous global")

    
    def save_metadata(self):
        """Save adapter manager metadata"""
        metadata = {
            "base_model": self.base_model_name,
            "users": self.users,
            "aggregation_history": self.aggregation_history,
            "last_updated": datetime.now().isoformat()
        }
        
        with open(os.path.join(self.base_model_dir, "adapter_metadata.json"), "w") as f:
            json.dump(metadata, f, indent=2)
    
    def load_metadata(self):
        """Load adapter manager metadata"""
        metadata_path = os.path.join(self.base_model_dir, "adapter_metadata.json")
        
        if os.path.exists(metadata_path):
            with open(metadata_path, "r") as f:
                metadata = json.load(f)
                
            self.users = metadata.get("users", {})
            self.aggregation_history = metadata.get("aggregation_history", [])
            logger.info(f"Loaded metadata for {len(self.users)} users")
            return True
        return False

    def blend_adapters(self, user_id, base_model,peft_config, blend_ratio=0.5, query_text=None, tokenizer=None, reference_embeddings=None):
        """
        Create a blended model using both local and global adapters.
        Now uses dynamic weighting based on similarity between the query and the user's training data.

        Args:
            user_id: User identifier
            base_model: Base model to adapt
            blend_ratio: Default ratio if no query or refs provided (0=global only, 1=local only)
            query_text: The text/query to compute similarity for
            tokenizer: Tokenizer to convert query_text into tokens
            reference_embeddings: Precomputed tensor of shape (N, D) for user's training data

        Returns:
            Blended model
        """

        adapter_paths = self.get_user_adapter_paths(user_id)

        # Load both adapters
        local_model = PeftModel.from_pretrained(base_model, adapter_paths["local"].replace("\\", "/"))
        global_model = PeftModel.from_pretrained(base_model, self.global_adapter_dir.replace("\\", "/"))

        local_state = local_model.state_dict()
        global_state = global_model.state_dict()

        # --- Dynamic Weighting Calculation ---
        if query_text is not None and tokenizer is not None and reference_embeddings is not None:
            # Tokenize query
            inputs = tokenizer(query_text, return_tensors="pt")
            inputs = {k: v.to(base_model.device) for k, v in inputs.items()}

            # Get query embedding from base model + global adapter
            with torch.no_grad():
                outputs = global_model.base_model(**inputs, output_hidden_states=True)
                query_embedding = outputs.hidden_states[-1][:, -1, :]  # final token embedding

            # Compute cosine similarity with reference embeddings
            sim_scores = cosine_similarity(query_embedding, reference_embeddings)
            avg_similarity = sim_scores.mean().item()  # scalar

            # Map similarity [0,1] → weight [0,1]
            dynamic_ratio = max(0.0, min(1.0, avg_similarity))
        else:
            # Fallback to provided blend_ratio
            dynamic_ratio = blend_ratio

        # --- Blend the adapters using dynamic_ratio ---
        blended_model = get_peft_model(copy.deepcopy(base_model), peft_config)
        blended_state = blended_model.state_dict()

        for key in blended_state:
            if "lora" in key and key in local_state and key in global_state:
                blended_state[key] = (
                    dynamic_ratio * local_state[key] +
                    (1 - dynamic_ratio) * global_state[key]
                )

        for name, param in blended_model.named_parameters():
            if "lora" in name and name in blended_state:
                param.data = blended_state[name].to(param.device)

        return blended_model


# Query_router

In [ ]:
import numpy as np
from sentence_transformers import util
import logging

logger = logging.getLogger(__name__)

class QueryRouter:
    def __init__(self, embedding_model, router_threshold=0.65):
        """
        Initialize a query router to direct queries to local or global adapter

        Args:
            embedding_model: SentenceTransformer model for embeddings
            router_threshold: Similarity threshold to route to local adapter
        """
        self.embedding_model = embedding_model
        self.router_threshold = router_threshold

    def route_query(self, query_text, statement_array, graph=None):
        """
        Route a query to either local or global adapter based on similarity
        to a provided array of interest statements.

        Args:
            query_text: The query to route
            statement_array: List of interest statements (preferences, past queries)
            graph: Optional preference graph to help with routing

        Returns:
            tuple: (adapter_type, relevance_score)
        """
        if not statement_array:
            logger.info("No interest statements provided, using global adapter")
            return "global", 0.0

        # Compute embedding for the query
        query_emb = self.embedding_model.encode([query_text])[0]

        # Compute embeddings for all interests
        interest_embs = self.embedding_model.encode(statement_array)

        # Calculate cosine similarities
        similarities = util.cos_sim(query_emb, interest_embs)[0].tolist()

        # Find the top matches
        top_k = min(5, len(similarities))
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        top_similarities = [similarities[i] for i in top_indices]
        top_interests = [statement_array[i] for i in top_indices]

        # Calculate a weighted relevance score
        relevance_score = max(similarities) if similarities else 0.0

        # Use graph information if available to boost relevance
        if graph:
            query_lower = query_text.lower()
            graph_genres = self._extract_graph_genres(graph)

            for genre in graph_genres:
                if genre.lower() in query_lower:
                    relevance_score = max(relevance_score, self.router_threshold + 0.1)
                    break

        # Make routing decision
        adapter_type = "local" if relevance_score >= self.router_threshold else "global"

        logger.info(f"Query: '{query_text}' routed to {adapter_type} adapter (relevance: {relevance_score:.4f})")
        if top_interests:
            logger.info(f"Top matching interests: {', '.join(top_interests[:3])}")

        # matching_info = [
        #     {"interest": interest, "similarity": float(sim)} 
        #     for interest, sim in zip(top_interests, top_similarities)
        # ]

        return adapter_type, relevance_score
    
    def _extract_graph_genres(self, graph):
        """Extract genres from preference graph"""
        genres = []
        for _, data in graph.nodes(data=True):
            if "genre" in data and data["genre"]:
                genres.extend(data["genre"])
        return list(set(genres))


# Query

In [ ]:
import json
from langchain_core.messages import HumanMessage
import re

def _call_llm_with_retry(llm, prompt_template, input_data: dict, max_chances: int = 3, operation_name: str = "process"):
    """Generic function to call LLM with retry logic."""
    chain = prompt_template | llm
    
    for attempt in range(max_chances):
        try:
            print(f"🔄 Attempt {attempt + 1} to {operation_name}...")
            result = chain.invoke(input_data)
            return result.content.strip()
        except Exception as e:
            print(f"⚠️ LLM call failed on attempt {attempt + 1}:", e)
            continue
    return None


def query_model(user_input, llm, prompt_template, **kwargs):
    """
    Query the LLM with the given prompt template.
    
    Args:
        user_input: User input (can be None if using other parameters)
        llm: LLM instance
        prompt_template: PromptTemplate instance
        **kwargs: Additional parameters for the prompt template
    
    Returns:
        The generated response
    """
    max_retries = 5
    
    # Build parameters dictionary
    params = {}
    if user_input is not None:
        params["user_input"] = user_input
    
    # Add any additional parameters from kwargs
    params.update(kwargs)
    
    # Loop through retries
    for attempt in range(1, max_retries + 1):
        try:
            print(f"🔄 Attempt {attempt} to classify user input...")
            
            # Format the prompt
            prompt = prompt_template.format(**params)
            
            # Call the LLM
            response = llm.invoke([HumanMessage(content=prompt)])
            return response.content
            
        except Exception as e:
            print(f"⚠️ LLM call failed on attempt {attempt}: {str(e)}")
            
            if attempt == max_retries:
                raise ValueError(f"Unable to classify input after multiple attempts.")


def extract_preferences(user_prompt: str, llm, prompt_template, length="medium", language="English") -> dict:
    result_str = _call_llm_with_retry(
        llm, 
        prompt_template, 
        {
            "user_prompt": user_prompt,
            "length": length,
            "language": language
        }, 
        1,
        "extract preferences"
    )
    
    if result_str is None:
        return {}

    # Strip markdown-style code blocks (```json ... ```
    cleaned_result = re.sub(r"```(?:json)?\n(.*)\n```", r"\1", result_str.strip(), flags=re.DOTALL)

    try:
        result_dict = json.loads(cleaned_result)
        return result_dict
    except json.JSONDecodeError:
        print(f"⚠️ Invalid JSON received:\n{cleaned_result}\n")

    # If parsing fails, return empty dict
    return {}


# Stemming

In [ ]:
from nltk.stem import PorterStemmer

# Create stemmer once at module level
stemmer = PorterStemmer()

def apply_stemming(data):
    """Apply stemming to a string or list of strings"""
    if isinstance(data, str):
        words = data.lower().split()
        return [stemmer.stem(word) for word in words]
    elif isinstance(data, list):
        stemmed_list = []
        for item in data:
            item_words = item.lower().split()
            stemmed_list.extend([stemmer.stem(word) for word in item_words])
        return stemmed_list
    return []


# CPL

In [ ]:
import json
import uuid
from datetime import datetime
import networkx as nx
import numpy as np
from transformers import BertTokenizer, BertModel
import torch
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.offline import plot
from sklearn.metrics.pairwise import cosine_similarity
import os

# ---------- Helpers (NEW / REFACTORED) ----------
def generate_embedding(text, tokenizer=None, model=None):
    """Generate BERT embedding for a given text."""
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :].detach().cpu().numpy().flatten()
    return embedding / np.linalg.norm(embedding)


def _lower(s):
    return s.lower() if isinstance(s, str) else s

def _normalize_to_list(v):
    if v is None:
        return []
    if isinstance(v, list):
        return [_lower(x) for x in v if isinstance(x, str)]
    if isinstance(v, str):
        return [_lower(v)]
    return []

def _stem_list(lst):
    return apply_stemming(lst) if lst else []

def _canonicalize_pref_types(pref_types):
    """Map synonymous preference types to canonical forms."""
    if not pref_types:
        return []
    mapping = {
        'like': 'prefer',
        'likes': 'prefer',
        'love': 'prefer',
        'enjoy': 'prefer',
        'prefer': 'prefer',
        'dislike': 'dislike',
        'hate': 'dislike',
        'avoid': 'avoid',
        'neutral': 'neutral'
    }
    out = []
    for p in pref_types:
        p = p.lower()
        out.append(mapping.get(p, p))
    # de-dup
    return sorted(set(out))

def _get_semantic_similarity(embedding1, embedding2):
    """
    Calculate cosine similarity between two embeddings.
    """
    if embedding1 is None or embedding2 is None:
        return 0.0
    return float(cosine_similarity([embedding1], [embedding2])[0][0])

def _dedupe_preferences(prefs, similarity_threshold=0.95):
    """
    Enhanced deduplication that considers both exact matches and semantic similarity.
    
    Args:
        prefs: List of preference dictionaries
        similarity_threshold: Threshold for considering preferences semantically similar
        
    Returns:
        List of deduplicated preferences
    """
    # First do exact deduplication (keep latest timestamp)
    before = len(prefs)
    prefs.sort(key=lambda x: x['timestamp'], reverse=True)
    seen = set()
    out = []
    for p in prefs:
        key = (p['statement'], tuple(sorted(p['genre'])), p['specific_item'])
        if key in seen:
            print(f"[DeDupe] Removed exact duplicate: {p['statement']}")
            continue
        seen.add(key)
        out.append(p)
    
    exact_deduped = len(out)
    if exact_deduped != before:
        print(f"[DeDupe] {before-exact_deduped} exact duplicate(s) removed")
    
    # Then check for semantic similarity if embeddings exist
    if any('embedding' in p and p['embedding'] is not None for p in out):
        semantic_deduped = []
        for i, p1 in enumerate(out):
            # If this preference has already been marked as a duplicate, skip
            if p1.get('is_duplicate', False):
                continue
                
            semantic_deduped.append(p1)
            p1_embedding = p1.get('embedding')
            if p1_embedding is None:
                continue
                
            # Look for semantically similar preferences
            for j in range(i+1, len(out)):
                p2 = out[j]
                if p2.get('is_duplicate', False):
                    continue
                    
                p2_embedding = p2.get('embedding')
                if p2_embedding is None:
                    continue
                    
                similarity = _get_semantic_similarity(p1_embedding, p2_embedding)
                
                # If genres match and statements are highly similar, mark as duplicate
                genres_overlap = bool(set(p1['genre']).intersection(set(p2['genre'])))
                if similarity > similarity_threshold and genres_overlap:
                    p2['is_duplicate'] = True
                    print(f"[DeDupe] Semantic duplicate ({similarity:.2f}): '{p2['statement']}' similar to '{p1['statement']}'")
        
        semantic_removed = len(out) - len(semantic_deduped)
        if semantic_removed > 0:
            print(f"[DeDupe] {semantic_removed} semantic duplicate(s) removed")
            out = semantic_deduped
    
    return out

def _standardize_preference(statement, preference_type, genre, specific_item, conditions,
                            timestamp=None, active=True):
    return {
        'statement': statement.lower(),
        'preference_type': _stem_list(_normalize_to_list(preference_type)),
        'genre': _stem_list(_normalize_to_list(genre)),
        'specific_item': specific_item.lower() if isinstance(specific_item, str) else None,
        'conditions': {k.lower(): _lower(v) for k, v in (conditions or {}).items()
                       if k.lower() != 'location'},
        'embedding': None,
        'timestamp': timestamp or datetime.now().isoformat(),
        'active': active
    }

# ---------- UPDATED: parse extracted structure ----------

def parse_extracted_preferences(extraction, original_prompt=None):
    """
    Parse JSON from extract_preferences_prompt (implicit preferences optionally ignored).

    Expected structure:
    {
      "explicit": [
        {
          "type": "prefer",
          "genre": "movie",
          "specific item": "comedy",
          "conditions": {"language": "english"},
          "statement": "User prefers comedy movies in English."
        }
      ],
      "implicit": {
        "statement": "User is in a relaxed mood."
      }
    }

    Parameters:
        extraction: dict | str
        original_prompt: fallback statement text
        include_implicit (bool): if True, also convert implicit statement into a preference node
    """
    if isinstance(extraction, str):
        extraction = json.loads(extraction)

    prefs = []
    explicit_list = extraction.get('explicit') or []
    for e in explicit_list:
        prefs.append(
            _standardize_preference(
                statement=e.get('statement') or (original_prompt or ""),
                preference_type=_normalize_to_list(e.get('type')),
                genre=_normalize_to_list(e.get('genre')),
                specific_item=e.get('specific item'),
                conditions=e.get('conditions') or {},
                timestamp=e.get('timestamp', datetime.now().isoformat()),
                active=True
            )
        )

    return _dedupe_preferences(prefs)

def parse_explicit_features(json_input):
    """
    Backward compatibility + direct support for new extract_preferences_prompt output.
    """
    data = json_input
    if isinstance(json_input, str):
        data = json.loads(json_input)

    # New format heuristic: contains keys 'explicit' or 'implicit'
    if isinstance(data, dict) and ('explicit' in data or 'implicit' in data):
        return parse_extracted_preferences(data)

    # Fallback old format (unchanged logic)
    print("Falling back to old format")
    preference_nodes = []
    if isinstance(data, list):
        data = {str(i): v for i, v in enumerate(data)}
    for prompt, details in data.items():
        explicit_features = details.get('explicit', [])
        for feature in explicit_features:
            flattened_conditions = {}
            for key, value in feature.get('conditions', {}).items():
                if key.lower() == 'location':
                    continue
                if isinstance(value, dict) and 'attribute' in value:
                    vval = value.get('value')
                    flattened_conditions[value['attribute'].lower()] = _lower(vval)
                else:
                    flattened_conditions[key.lower()] = _lower(value)
            genres = feature.get('genre', [])
            genres = _stem_list(_normalize_to_list(genres))
            preference_types = feature.get('type', [])
            preference_types = _stem_list(_normalize_to_list(preference_types))
            preference_nodes.append(
                _standardize_preference(
                    statement=prompt,
                    preference_type=preference_types,
                    genre=genres,
                    specific_item=feature.get('specific item'),
                    conditions=flattened_conditions,
                    timestamp=feature.get('timestamp', datetime.now().isoformat()),
                    active=True
                )
            )
    return _dedupe_preferences(preference_nodes)

# ---------- Conflict Resolution (unchanged logic, refactored) ----------

def resolve_preference_conflicts(preference_nodes, tokenizer=None, model=None):
    """
    Enhanced conflict resolution that handles:
    1. Direct contradictions (same item/genre with opposite preference types)
    2. Logical contradictions (cannot both prefer and dislike the same item)
    3. Time-based resolution (newer preferences override older ones)
    4. Semantic similarity detection (using embeddings if available)
    
    Args:
        preference_nodes: List of preference dictionaries
        tokenizer: BERT tokenizer for embedding generation if needed
        model: BERT model for embedding generation if needed
        
    Returns:
        List of preferences with conflicts resolved
    """
    changed = False
    preference_nodes.sort(key=lambda x: x['timestamp'], reverse=True)
    
    # Generate embeddings for preferences that don't have them
    if tokenizer and model:
        for pref in preference_nodes:
            if pref.get('embedding') is None:
                pref['embedding'] = generate_embedding(pref['statement'], tokenizer, model)
    
    # First pass: Handle direct contradictions (same item, opposite preference)
    for i, pref in enumerate(preference_nodes):
        if not pref['active']:
            continue
            
        # Get the canonical preference types
        pref_types = _canonicalize_pref_types(pref['preference_type'])
        
        for j, other_pref in enumerate(preference_nodes[i+1:], i+1):
            if not other_pref['active']:
                continue
                
            other_pref_types = _canonicalize_pref_types(other_pref['preference_type'])
            
            # Check for direct contradictions
            same_item = (pref['specific_item'] is not None and 
                         pref['specific_item'] == other_pref['specific_item'])
                         
            genre_overlap = bool(set(pref['genre']).intersection(set(other_pref['genre'])))
            
            # Detect opposing preference types
            opposing_prefs = False
            if ('prefer' in pref_types and 'dislike' in other_pref_types) or \
               ('dislike' in pref_types and 'prefer' in other_pref_types):
                opposing_prefs = True
            
            # If we have the same item or overlapping genres with opposing preferences, 
            # deactivate the older one
            if (same_item or genre_overlap) and opposing_prefs:
                other_pref['active'] = False
                changed = True
                print(f"[Conflict] Deactivated contradictory preference:"
                      f" '{other_pref['statement']}'"
                      f" (conflicts with: '{pref['statement']}')")
    
    # Second pass: Handle semantic contradictions (if embeddings available)
    if tokenizer and model:
        for i, pref in enumerate(preference_nodes):
            if not pref['active'] or pref.get('embedding') is None:
                continue
                
            pref_types = _canonicalize_pref_types(pref['preference_type'])
            
            for j, other_pref in enumerate(preference_nodes[i+1:], i+1):
                if not other_pref['active'] or other_pref.get('embedding') is None:
                    continue
                    
                other_pref_types = _canonicalize_pref_types(other_pref['preference_type'])
                
                # Check for semantic similarity
                similarity = _get_semantic_similarity(pref['embedding'], other_pref['embedding'])
                
                # If semantically similar with genre overlap and opposing preferences
                genre_overlap = bool(set(pref['genre']).intersection(set(other_pref['genre'])))
                
                # Detect opposing preference types
                opposing_prefs = False
                if ('prefer' in pref_types and 'dislike' in other_pref_types) or \
                   ('dislike' in pref_types and 'prefer' in other_pref_types):
                    opposing_prefs = True
                
                if similarity > 0.7 and genre_overlap and opposing_prefs:
                    other_pref['active'] = False
                    changed = True
                    print(f"[Conflict] Deactivated semantically contradictory preference:"
                          f" '{other_pref['statement']}'"
                          f" (conflicts with: '{pref['statement']}', similarity: {similarity:.2f})")
    
    if changed:
        print("[Conflict] Conflict resolution completed with deactivations.")
    return preference_nodes

def resolve_preference_conflicts_in_graph(graph, new_pref, tokenizer=None, model=None):
    """
    Reuse resolve_preference_conflicts logic on combined list,
    then propagate 'active' flags back to graph for older conflicting nodes.
    
    Args:
        graph: NetworkX graph containing preferences
        new_pref: New preference to add
        tokenizer: BERT tokenizer for embedding generation
        model: BERT model for embedding generation
        
    Returns:
        Updated new_pref after conflict resolution
    """
    existing = []
    for node, data in graph.nodes(data=True):
        if data.get('type') == 'Preference':
            existing.append({
                'node_id': node,
                'statement': data['statement'],
                'preference_type': data['preference_type'],
                'genre': data['genre'],
                'specific_item': data['specific_item'],
                'embedding': data.get('embedding'),
                'timestamp': data['timestamp'],
                'active': data['active']
            })
    
    # Generate embedding for new preference if not already present
    if new_pref.get('embedding') is None and tokenizer and model:
        new_pref['embedding'] = generate_embedding(new_pref['statement'], tokenizer, model)
    
    combo = existing + [new_pref]
    
    # Apply same ordering & conflict rules - pass tokenizer and model to the function
    resolve_preference_conflicts(combo, tokenizer, model)
    
    # Push deactivations to graph
    for p in combo:
        if 'node_id' in p:
            if graph.nodes[p['node_id']]['active'] != p['active']:
                graph.nodes[p['node_id']]['active'] = p['active']
                print(f"[Graph] Updated node {p['node_id']} active status to {p['active']}")
                
                # Update edge active status as well
                for _, v, ed in graph.out_edges(p['node_id'], data=True):
                    ed['active'] = p['active']
                for u, _, ed in graph.in_edges(p['node_id'], data=True):
                    ed['active'] = p['active']
                    
    return new_pref

# ---------- UPDATED: append preferences ----------

def append_features_in_the_graph(graph, new_preferences, tokenizer=None, model=None):
    """
    Append standardized preference dicts (output of parse_extracted_preferences / parse_explicit_features).
    """
    # Coerce single dict to list
    if isinstance(new_preferences, dict):
        new_preferences = [new_preferences]
        
    # Generate embeddings for all preferences
    for pref in new_preferences:
        if pref.get('embedding') is None and tokenizer and model:
            pref['embedding'] = generate_embedding(pref['statement'], tokenizer=tokenizer, model=model)
    
    # Apply semantic deduplication before adding to graph
    new_preferences = _dedupe_preferences(new_preferences)
    
    user_id = next(n for n, d in graph.nodes(data=True) if d['type'] == 'User')
    
    # Resolve conflicts within the new preferences
    new_preferences = resolve_preference_conflicts(new_preferences, tokenizer, model)

    for pref in new_preferences:
        # Skip if marked as duplicate or inactive
        if pref.get('is_duplicate', False) or not pref['active']:
            continue
            
        # Check for exact duplicates in graph
        key = (pref['statement'], tuple(sorted(pref['genre'])), pref['specific_item'])
        remove_node_id = None
        skip_pref = False
        
        for node, data in graph.nodes(data=True):
            if data.get('type') == 'Preference':
                existing_key = (data['statement'], tuple(sorted(data['genre'])), data['specific_item'])
                
                # Check for exact match
                if key == existing_key:
                    if data['timestamp'] >= pref['timestamp']:
                        skip_pref = True
                        print(f"[Skip] Existing preference newer or same timestamp: {key}")
                    else:
                        remove_node_id = node
                        print(f"[Replace] Incoming preference is newer, replacing node {node}: {key}")
                    break
                
                # Check for semantic similarity if embeddings exist
                if tokenizer and model and not skip_pref and not remove_node_id:
                    existing_embedding = data.get('embedding')
                    if existing_embedding is not None and pref.get('embedding') is not None:
                        similarity = _get_semantic_similarity(pref['embedding'], existing_embedding)
                        
                        # If very similar with overlapping genres, check timestamp
                        if similarity > 0.9 and bool(set(pref['genre']).intersection(set(data['genre']))):
                            if data['timestamp'] >= pref['timestamp']:
                                skip_pref = True
                                print(f"[Skip] Semantically similar preference exists: {similarity:.2f} similar to '{data['statement']}'")
                            else:
                                remove_node_id = node
                                print(f"[Replace] Replacing semantically similar preference: {similarity:.2f} similar to '{data['statement']}'")
                            break
        
        if skip_pref:
            continue
            
        if remove_node_id:
            graph.remove_node(remove_node_id)

        # Resolve conflicts with graph
        pref = resolve_preference_conflicts_in_graph(graph, pref, tokenizer, model)
        
        # Skip if preference became inactive during conflict resolution
        if not pref['active']:
            continue

        # Add the preference to the graph
        pref_id = str(uuid.uuid4())
        graph.add_node(pref_id, type='Preference',
                       statement=pref['statement'],
                       preference_type=pref['preference_type'],
                       genre=pref['genre'],
                       specific_item=pref['specific_item'],
                       conditions=pref['conditions'],
                       embedding=pref.get('embedding'),
                       timestamp=pref['timestamp'],
                       active=pref['active'])
        graph.add_edge(user_id, pref_id, relation='has_preference', active=pref['active'])
        
        key_display = (pref['statement'], pref['genre'], pref['specific_item'])
        print(f"[Add] Added preference node {pref_id}: {key_display}")

        item_name = pref['specific_item'] or (pref['genre'][0] if pref['genre'] else "unknown")
        item_id = f"item_{item_name.replace(' ', '_')}_{uuid.uuid4().hex[:8]}"
        graph.add_node(item_id, type='Item', name=item_name, genre=pref['genre'])
        graph.add_edge(pref_id, item_id, relation='relates_to',
                       relation_type=pref['preference_type'],
                       active=pref['active'])

        tag_values = []
        if pref['specific_item']:
            tag_values.append(pref['specific_item'])
        tag_values.extend(pref['conditions'].values())
        tag_values.extend(pref['genre'])
        for tag in set(tag_values):
            tag_id = f"tag_{tag.replace(' ', '_')}"
            if not graph.has_node(tag_id):
                graph.add_node(tag_id, type='Tag', name=tag)
            graph.add_edge(item_id, tag_id, relation='has_tag', active=pref['active'])
    
    return graph


# ---------- UPDATED: query_preferences (optional query node) ----------

def query_preferences(graph, user_id, query_text, tokenizer=None, model=None,
                      genre_filter=None, preference_type_filter=None,
                      top_k=5, active_only=True, relevance_threshold=0.4):
    """Enhanced preference querying with better cross-genre conceptual matching"""
    
    if genre_filter is not None and not isinstance(genre_filter, list):
        genre_filter = [genre_filter]
    if preference_type_filter is not None and not isinstance(preference_type_filter, list):
        preference_type_filter = [preference_type_filter]

    # Canonicalize preference types (handle 'like' -> 'prefer')
    preference_type_filter = _canonicalize_pref_types(preference_type_filter)

    # Generate query embedding
    query_embedding = generate_embedding(query_text.lower(), tokenizer=tokenizer, model=model) if tokenizer and model else None
    
    # Apply stemming to genre filter if provided
    stemmed_genre_filter = _stem_list(genre_filter) if genre_filter else None
    
    # First gather ALL candidates without genre filtering
    all_candidates = []
    for node, data in graph.nodes(data=True):
        if data.get('type') != 'Preference':
            continue
        if not graph.has_edge(user_id, node):
            continue
        if active_only and not data['active']:
            continue
            
        # Calculate relevance factors for ALL preferences
        relevance_factors = {}
        
        # 1. Semantic similarity (most important for cross-concept matching)
        semantic_sim = 0.0
        if query_embedding is not None and data.get('embedding') is not None:
            semantic_sim = float(np.dot(data['embedding'], query_embedding))
            relevance_factors['semantic_similarity'] = semantic_sim
        
        # 2. Keyword matching
        statement_terms = set(_stem_list([w.lower() for w in data['statement'].split()]))
        query_terms = set(_stem_list([w.lower() for w in query_text.split()]))
        keyword_overlap = len(query_terms.intersection(statement_terms)) / max(1, len(query_terms))
        relevance_factors['keyword_match'] = keyword_overlap
        
        # 3. Genre matching - MORE NUANCED APPROACH
        # Consider preferences with no genre as potentially relevant to any query
        if not data['genre']: 
            # No genre specified = potential match with anything, give moderate score
            relevance_factors['genre_match'] = 0.7
            
            # Check if any terms in the preference match the genre filter
            if stemmed_genre_filter and any(term in statement_terms for term in stemmed_genre_filter):
                relevance_factors['genre_match'] = 0.9  # Boost score if statement contains genre terms
        else:
            # With genres specified, check for exact or semantic matches
            if stemmed_genre_filter:
                # Direct genre match
                exact_match = any(any(gf in g for g in data['genre']) for gf in stemmed_genre_filter)
                if exact_match:
                    relevance_factors['genre_match'] = 1.0
                else:
                    # Even without direct match, calculate partial similarity
                    # Look for semantic relationships between genres if available
                    relevance_factors['genre_match'] = 0.5  # Base score for having any genre
            else:
                # No filter specified but preference has genres
                relevance_factors['genre_match'] = 0.8
        
        # Calculate weighted relevance score
        weights = {
            'semantic_similarity': 0.5,     # Keep semantic similarity as most important
            'keyword_match': 0.2,           # Reduce slightly
            'genre_match': 0.2,             # New factor with significant weight
            'recency': 0.1
        }
        
        # Ensure all factors have values
        for factor in weights:
            if factor not in relevance_factors:
                relevance_factors[factor] = 0.0
                
        relevance_score = sum(weights[factor] * relevance_factors[factor] 
                             for factor in weights if factor in relevance_factors)
        
        all_candidates.append((node, data, relevance_score, relevance_factors))
    
    # Filter by threshold
    candidates = [c for c in all_candidates if c[2] >= relevance_threshold]

    if not candidates:
        print("[Query] Found 0 potential matches, returning top 0")
        return []

    # Sort and select top candidates
    candidates.sort(key=lambda x: x[2], reverse=True)
    top_candidates = candidates[:min(top_k, len(candidates))]

    # Debug logging
    print(f"[Query] Found {len(candidates)} potential matches, returning top {len(top_candidates)}")
    for i, (_, data, score, factors) in enumerate(top_candidates):
        print(f"[Query] #{i+1} [{score:.3f}] {data['statement'][:50]}... (factors: {', '.join([f'{k}={v:.2f}' for k, v in factors.items()])})")

    # Format results
    context = []
    for pref_id, data, score, _ in top_candidates:
        items = [graph.nodes[i]['name'] for _, i, e in graph.out_edges(pref_id, data=True)
                 if e['relation'] == 'relates_to' and (not active_only or e['active'])]
        context.append({
            'statement': data['statement'],
            'items': ", ".join(items) if items else "",
            'active': data['active'],
            'timestamp': data['timestamp'],
            'genre': ", ".join(data['genre']),
            'preference_type': ", ".join(data['preference_type']),
            'relevance_score': float(score)  # Add relevance score to output
        })

    return context

# ---------- UPDATED: visualize_interactive_graph ----------

def visualize_interactive_graph(graph, user_id, output_folder="./kaggle/working/visualization"):
    """
    Visualize the graph and save the interactive HTML file in the specified folder
    with the filename as user_id.html. Creates the folder if it does not exist.

    Args:
        graph (nx.Graph): The graph to visualize.
        user_id (str): The user ID to use as the filename.
        output_folder (str): The directory where the visualization should be saved.
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    output_file = os.path.join(output_folder, f"{user_id}.html")

    pos = nx.spring_layout(graph, dim=3, k=0.5, iterations=50)
    
    node_x, node_y, node_z = [], [], []
    node_colors, node_labels, node_types = [], [], []
    node_sizes = []
    
    for node, data in graph.nodes(data=True):
        x, y, z = pos[node]
        node_x.append(x)
        node_y.append(y)
        node_z.append(z)
        node_type = data['type']
        node_types.append(node_type)
        
        if node_type == 'User':
            node_colors.append('blue')
            node_labels.append(node)
            node_sizes.append(15)
        elif node_type == 'Preference':
            node_colors.append('green' if data['active'] else 'lightgray')
            label = data['statement'][:20] + '...' if len(data['statement']) > 20 else data['statement']
            node_labels.append(f"{label} ({data['timestamp'][:10]})")
            node_sizes.append(10)
        elif node_type == 'Item':
            node_colors.append('red')
            node_labels.append(data['name'])
            node_sizes.append(8)
        elif node_type == 'Tag':
            node_colors.append('yellow')
            node_labels.append(data['name'])
            node_sizes.append(6)
    
    # Create separate edge traces for individual opacity control
    edge_traces = []
    for u, v, data in graph.edges(data=True):
        x0, y0, z0 = pos[u]
        x1, y1, z1 = pos[v]
        edge_color = ('blue' if data['relation'] == 'has_preference' else
                      'red' if data['relation'] == 'relates_to' else
                      'green' if data['relation'] == 'has_tag' else 'gray')
        edge_opacity = 1.0 if data['active'] else 0.3
        edge_traces.append(go.Scatter3d(
            x=[x0, x1, None],
            y=[y0, y1, None],
            z=[z0, z1, None],
            mode='lines',
            line=dict(color=edge_color, width=2),
            opacity=edge_opacity,
            hoverinfo='none',
            name=f'Edge {u}-{v}'
        ))
    
    node_trace = go.Scatter3d(
        x=node_x, y=node_y, z=node_z,
        mode='markers+text',
        marker=dict(size=node_sizes, color=node_colors, opacity=1.0),
        text=node_labels,
        textposition='top center',
        hoverinfo='text',
        customdata=[(node, node_type) for node, node_type in zip(graph.nodes, node_types)],
        name='Nodes'
    )
    
    layout = go.Layout(
        title='Interactive Preference Graph (Click to Highlight, Drag to Rotate)',
        scene=dict(
            xaxis=dict(showbackground=False),
            yaxis=dict(showbackground=False),
            zaxis=dict(showbackground=False),
            dragmode='orbit'
        ),
        showlegend=True,
        hovermode='closest',
        margin=dict(l=0, r=0, b=0, t=40)
    )
    
    fig = go.Figure(data=edge_traces + [node_trace], layout=layout)
    
    js_code = """
    function(e) {
        var plot = document.getElementsByClassName('plotly')[0];
        var node_data = e.points[0].customdata[0];
        var graph = %s;
        var node_trace = plot.data[plot.data.length - 1];
        var edge_traces = plot.data.slice(0, plot.data.length - 1);
        
        var node_opacities = new Array(node_trace.x.length).fill(0.2);
        var node_idx = node_trace.customdata.findIndex(d => d[0] === node_data);
        node_opacities[node_idx] = 1.0;
        
        var edge_opacities = new Array(edge_traces.length).fill(0.2);
        graph.forEach(function(edge, idx) {
            if (edge[0] === node_data || edge[1] === node_data) {
                edge_opacities[idx] = edge[2].active ? 1.0 : 0.3;
                var other_node = edge[0] === node_data ? edge[1] : edge[0];
                var other_idx = node_trace.customdata.findIndex(d => d[0] === other_node);
                node_opacities[other_idx] = 1.0;
            }
        });
        
        var updates = edge_opacities.map(function(op) { return {'opacity': op}; });
        Plotly.restyle(plot, updates, Array.from({length: edge_traces.length}, (_, i) => i));
        Plotly.restyle(plot, {'marker.opacity': node_opacities}, [plot.data.length - 1]);
    }
    """ % json.dumps([(u, v, {'active': d['active']}) for u, v, d in graph.edges(data=True)])
    
    fig.data[-1].on_click(js_code, True)
    
    plot(fig, filename=output_file, auto_open=True)
    print(f"Visualization saved to {output_file}")

def build_preference_graph(user_id):
    """
    Build initial preference graph with just the user node.
    
    Args:
        user_id (str): User identifier.
    
    Returns:
        nx.DiGraph: Directed graph with only the user node.
    """
    G = nx.DiGraph()
    G.add_node(user_id, type='User')
    return G



# Implicit

In [ ]:
import librosa
import parselmouth
import numpy as np
from typing import List, Dict
from torch.utils.data import DataLoader
from datasets import Dataset
import logging
import torch
import json
from peft import PeftModel



# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def voice_to_text(audio_path: str, whisper_model) -> str:
    try:
        result = whisper_model.transcribe(audio_path, language="en")
        return result["text"].strip()
    except Exception as e:
        logger.error(f"Voice-to-text conversion failed: {e}")
        return ""

def extract_prosody_features(audio_path: str, whisper_model) -> Dict[str, float]:
    """
    Extract and normalize prosody features from audio:
    - mean pitch
    - pitch std
    - RMS energy
    - speaking rate

    Returns normalized values.
    """
    try:
        y, sr = librosa.load(audio_path)
        snd = parselmouth.Sound(audio_path)

        # Pitch extraction with filtering
        pitch = snd.to_pitch()
        pitch_values = pitch.selected_array['frequency']
        pitch_values = pitch_values[(pitch_values > 50) & (pitch_values < 500)]  # Clean pitch range

        mean_pitch = np.mean(pitch_values) if len(pitch_values) > 0 else 0.0
        std_pitch = np.std(pitch_values) if len(pitch_values) > 0 else 0.0

        # Energy (RMS)
        rms = librosa.feature.rms(y=y)[0]
        mean_rms = np.mean(rms) if len(rms) > 0 else 0.0

        # Duration and speaking rate
        duration = librosa.get_duration(y=y, sr=sr)
        text = voice_to_text(audio_path, whisper_model=whisper_model)
        word_count = len(text.split()) if text else 0
        speaking_rate = word_count / duration if duration > 0 else 0.0

        # Normalization (approx ranges; tweak if needed)
        mean_pitch_norm = (mean_pitch - 100) / 100     # assuming 100–300 Hz range
        std_pitch_norm = std_pitch / 50                # assuming max 50 Hz std
        mean_rms_norm = (mean_rms - 0.01) / 0.02        # assuming range 0.01–0.03
        speaking_rate_norm = (speaking_rate - 2) / 2    # assuming normal 2 wps

        return {
            "mean_pitch": float(mean_pitch_norm),
            "std_pitch": float(std_pitch_norm),
            "mean_rms": float(mean_rms_norm),
            "speaking_rate": float(speaking_rate_norm)
        }

    except Exception as e:
        logger.error(f"Prosody feature extraction failed: {e}")
        return {
            "mean_pitch": 0.0,
            "std_pitch": 0.0,
            "mean_rms": 0.0,
            "speaking_rate": 0.0
        }

def extract_implicit_features(text: str, embedding_model, sentiment_pipeline, prosody: Dict[str, float]) -> torch.Tensor:
    """
    Merge sentiment + embedding + prosody features into a single tensor.
    """
    try:
        logger.info(f"Processed text: {text}")

        # Sentiment
        sentiment = sentiment_pipeline(text)[0]
        sentiment_score = sentiment["score"]
        sentiment_label = sentiment["label"]
        label_to_id = {"NEGATIVE": 0, "NEUTRAL": 1, "POSITIVE": 2}
        sentiment_label_id = label_to_id.get(sentiment_label.upper(), 1)

        # Embedding
        embedding = embedding_model.encode([text])[0]

        # Merge all
        prosody_values = [
            prosody.get("mean_pitch", 0.0),
            prosody.get("std_pitch", 0.0),
            prosody.get("mean_rms", 0.0),
            prosody.get("speaking_rate", 0.0),
        ]

        merged_embedding = torch.tensor(
            [sentiment_score, sentiment_label_id] + prosody_values + list(embedding),
            dtype=torch.float32
        )
        return merged_embedding

    except Exception as e:
        logger.error(f"Implicit feature extraction failed: {e}")
        return torch.tensor([], dtype=torch.float32)

def update_lora_weights(texts: List[str], model, output_dir: str, embedding_model, sentiment_pipeline, prosody_list: List[Dict[str, float]], user_id: str = None, adapter_type: str = "local"):

    try:
        # Attach adapter to the base model
        model_peft = PeftModel.from_pretrained(model, output_dir.replace("\\", "/"))

        dataset = Dataset.from_dict({"text": texts})
        dataloader = DataLoader(dataset, batch_size=1)

        for i, batch in enumerate(dataloader):
            text = batch["text"][0]
            prosody = prosody_list[i] if i < len(prosody_list) else {}
            
            merged_features = extract_implicit_features(
                text, embedding_model, sentiment_pipeline, prosody
            )
            logger.info(f"Merged feature vector shape: {merged_features.shape}")

            try:
                for name, param in model_peft.named_parameters():
                    if "lora" in name and param.requires_grad:
                        if adapter_type == "local" or user_id is None:
                            param.data += 0.001 * merged_features.mean()
                        elif adapter_type == "global" and user_id is not None:
                            param.data += 0.0005 * merged_features.mean()
            except Exception as e:
                logger.error(f"LoRA update failed: {e}")

        # Save only adapter (not the whole model)
        model_peft.save_pretrained(output_dir.replace("\\", "/"), safe_serialization=True)
        logger.info(f"LoRA {adapter_type} adapter updated and saved to {output_dir}")

    except Exception as e:
        logger.error(f"LoRA weight update loop failed: {e}")

def Implicit_main(input_data, whisper_model, model, embedding_model, sentiment_pipeline, 
                 voice_metadata: str = None, user_id: str = None, adapter_manager: AdapterManager = None):
    try:
        # Convert single string to list if needed
        texts = [input_data] if isinstance(input_data, str) else input_data
        
        if not texts or not all(isinstance(t, str) for t in texts):
            logger.error(f"Invalid input: expected string or list of strings, got {type(input_data)}")
            return
            
        logger.info(f"Processing {len(texts)} text entries")
        
        # Prepare prosody features list (one per text)
        prosody_list = []
        
        # Process each text entry
        for i, text in enumerate(texts):
            # Check if we have voice recording for this text
            has_voice = False
            
            if voice_metadata:
                # voice_metadata is now a string (the path), not a dict
                voice_path = voice_metadata
                logger.info(f"Using provided voice recording for text '{text[:30]}...': {voice_path}")
                try:
                    prosody = extract_prosody_features(voice_path, whisper_model)
                    prosody_list.append(prosody)
                    logger.info(f"Extracted prosody features: {prosody}")
                    has_voice = True
                except Exception as e:
                    logger.error(f"Failed to extract prosody features from {voice_path}: {e}")
                    # Add default prosody as fallback
                    prosody_list.append({
                        "mean_pitch": 0.0,
                        "std_pitch": 0.0,
                        "mean_rms": 0.0,
                        "speaking_rate": 0.0
                    })
            
            # If no voice recording found or processing failed, add default prosody
            if not has_voice:
                logger.info(f"No voice data for text '{text[:30]}...', using default prosody")
                prosody_list.append({
                    "mean_pitch": 0.0,
                    "std_pitch": 0.0,
                    "mean_rms": 0.0,
                    "speaking_rate": 0.0
                })
        
        # Ensure we have prosody features for each text
        if len(prosody_list) != len(texts):
            logger.warning(f"Mismatch between texts ({len(texts)}) and prosody features ({len(prosody_list)})")
            # Extend with defaults if needed
            while len(prosody_list) < len(texts):
                prosody_list.append({
                    "mean_pitch": 0.0,
                    "std_pitch": 0.0,
                    "mean_rms": 0.0,
                    "speaking_rate": 0.0
                })
        
        # If we have an adapter manager and user_id, use it
        if adapter_manager and user_id:
            adapter_paths = adapter_manager.get_user_adapter_paths(user_id)
            print("@ User Adapter Path: ",adapter_paths)
            
            # Update the local adapter (implicit features only affect local)
            update_lora_weights(
                texts, model, adapter_paths["local"], 
                embedding_model, sentiment_pipeline, prosody_list,
                user_id=user_id, adapter_type="local"
            )
            
            logger.info(f"Updated local adapter for user {user_id} with implicit features")
        else:
            logger.info("Cannot do LoRA update, adapter_manager or user_id is not passed")

    except Exception as e:
        logger.error(f"Implicit_main function failed: {e}", exc_info=True)


if __name__ == "__main__":
    pass


# Util Function

In [ ]:
import re
import json
import os

def is_input_question(text):
    """
    Detect if input is likely a question rather than a preference statement.
    Returns True if it's a question, False otherwise.
    """
    # Check for question mark
    if '?' in text:
        return True
        
    # Check for common question words at the beginning
    question_starters = ['which', 'what', 'how', 'where', 'when', 'who', 'why', 'is', 'are', 'can', 'do', 'does', 'would', 'will', 'should', 'could']
    words = text.lower().strip().split()
    if words and words[0] in question_starters:
        return True
        
    # Check for inverted sentence structure (common in questions)
    # "Would I", "Do I", "Should I", etc.
    for starter in ['would', 'will', 'could', 'can', 'do', 'does', 'is', 'are', 'should', 'might']:
        if f"{starter} i" in text.lower() or f"{starter} you" in text.lower():
            return True
    
    return False


def extract_answer(response):
    response_lower = response.lower()
    clean_response = re.sub(r'[^\w\s]', '', response_lower).strip()
    return clean_response


def save_chat_history(chats, user_id):
    """Save chat history to a JSON file inside user_chats directory"""
    try:
        dir_name = "user_chats"
        if not os.path.exists(dir_name):
            os.makedirs(dir_name)
        filename = os.path.join(dir_name, f"chat_history_{user_id}.json")
        with open(filename, "w") as f:
            json.dump(chats, f, indent=2)
        print(f"Chat history saved to {filename}")
    except Exception as e:
        print(f"Error saving chat history: {e}")


def load_chat_history(user_id):
    """Load chat history from a JSON file if it exists"""
    dir_name = "user_chats"
    filename = os.path.join(dir_name, f"chat_history_{user_id}.json")
    if os.path.exists(filename):
        try:
            with open(filename, "r") as f:
                return json.load(f)
        except Exception as e:
            print(f"Error loading chat history: {e}")
    return {}


def run_implicit_feature_extraction(chat_obj, llm, model, sentiment_pipeline, embedding_model, whisper_model, user_id, adapter_manager):
    """
    Extract implicit features from a single chat object and process them.

    Args:
        chat_obj: Dict with keys "chat", "voice" (voice can be None)
        user_id: User identifier
    """
    try:
        # Use the already well-structured prompt from chat_obj["chat"]
        prompt = chat_obj.get("chat", "")

        # Get implicit features using the prompt template
        response_text = query_model(
            user_input=None,
            llm=llm,
            prompt_template=extract_implicit_features_prompt,
            chat_session=prompt
        )

        # Try to parse the response as JSON
        features_list = None
        try:
            features_list = json.loads(response_text)
        except json.JSONDecodeError:
            # Try to extract the JSON part from the string if it contains an error message
            if "Failed to parse response as JSON:" in response_text:
                # Extract the part after the colon
                json_start = response_text.find(":") + 1
                possible_json = response_text[json_start:].strip()
                try:
                    features_list = json.loads(possible_json)
                except Exception as e:
                    print(f"Still failed to parse extracted JSON: {e}")
                    print(f"Raw extracted string: {possible_json}")
                    return
            else:
                print(f"Failed to parse response as JSON: {response_text}")
                return

        if isinstance(features_list, list):
            prompt_texts = []
            for feature in features_list:
                if "Standard_fine_tuning_prompt" in feature:
                    prompt_texts.append(feature["Standard_fine_tuning_prompt"])

            if prompt_texts:
                print(f"Extracted {len(prompt_texts)} implicit features")
                print("#############################################")
                for pt in prompt_texts:
                    print(f" - {pt}")

                # Prepare voice metadata if available
                voice_metadata = None
                if chat_obj.get("voice"):
                    voice_metadata = chat_obj["voice"]

                Implicit_main(
                    input_data=prompt_texts,
                    whisper_model=whisper_model,
                    model=model,
                    embedding_model=embedding_model,
                    sentiment_pipeline=sentiment_pipeline,
                    voice_metadata=voice_metadata,
                    user_id=user_id,
                    adapter_manager=adapter_manager
                )
                return
        else:
            print(f"Unexpected response format: {response_text}")

    except Exception as e:
        print(f"Error extracting implicit features: {e}")


# Model Loading

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import os
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
from transformers import BertTokenizer, BertModel

import logging
import re
import json
import whisper
import pandas as pd
import torch
import warnings
from datetime import datetime


# Suppress warnings
warnings.filterwarnings("ignore")
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# Set up minimal logging
logging.basicConfig(level=logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

logger = logging.getLogger(__name__)


# Initialize Groq LLaMA-3 LLM
llm1 = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name=LLM_MODEL,
)
llm2 = ChatGroq(
    api_key=GROQ_API_KEY_2,
    model_name=LLM_MODEL,
)

# Set device to CPU explicitly
device = torch.device("cpu")
torch.set_default_device(device)

# Initialize models
# HF_TOKEN is loaded from Kaggle Secrets in the API keys cell above.
llama_model_name = "meta-llama/Llama-2-7b-chat-hf"
llama_torch_dtype = torch.float16

# estimate max memory per device (adjust if you know exact numbers)
max_mem = {
    0: "15GiB",
    1: "15GiB",
}

model = AutoModelForCausalLM.from_pretrained(
    llama_model_name,
    token = HF_TOKEN,
    torch_dtype=llama_torch_dtype,
    device_map="auto",
    max_memory=max_mem,
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(
    llama_model_name,
    token=HF_TOKEN,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)


# model.train()

# Initialize other models
whisper_model = whisper.load_model("base")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
sentiment_pipeline = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest")

# Load BERT tokenizer and model for graph building
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')
bert_model.eval()

print("Models loaded successfully!")


# Main

In [ ]:
adapter_manager = AdapterManager(base_model_name=llama_model_name)
adapter_manager.load_metadata()

def clean_llm_answer(raw_answer: str) -> str:
    """
    Cleans the raw LLM answer by removing echoed system prompts or tags.
    Keeps only the assistant's actual response.
    """
    if not raw_answer:
        return ""

    # 1. Remove <|system|>...</|system|> blocks if present
    cleaned = re.sub(r"<\|system\|>.*?</\|/?system\|>", "", raw_answer, flags=re.DOTALL)

    # 2. Remove stray <|assistant|> or <|user|> markers if model echoed them
    cleaned = re.sub(r"<\|/?(assistant|user)\|>", "", cleaned, flags=re.IGNORECASE)

    # 3. Normalize whitespace
    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    return cleaned

# Add this function to perform federated learning
def perform_federated_learning(base_model, peft_config, participating_users=None):
    """
    Perform federated learning to aggregate adapters
    
    Args:
        base_model: The base model
        peft_config: LoRA configuration
        participating_users: List of user IDs to include (None = all users)
        
    Returns:
        Updated global model
    """
    print("🌐 Starting federated learning process...")
    
    # Perform the aggregation
    global_model = adapter_manager.federated_aggregation(
        base_model, 
        peft_config, 
        participating_users
    )
    
    # Save metadata
    adapter_manager.save_metadata()
    
    print("✅ Federated learning complete!")
    return global_model

# Initialize the router after loading models
query_router = None

# Add at module level
user_routers = {}

In [ ]:
def adapter_decider(user_id, statement_array, user_input, graph=None):
    # Get both models
    local_model, global_model = adapter_manager.initialize_user(
        user_id, model, peft_config
    )
    
    # Determine routing
    adapter_type, relevance_score = query_router.route_query(user_input, statement_array, graph)
    
    print(f"🧭 Query relevance score: {relevance_score:.4f}")
    
    # Use advanced blending for borderline cases
    if 0.45 <= relevance_score <= 0.85:
        # Calculate blend ratio (0.45->0.0, 0.65->0.5, 0.85->1.0)
        blend_ratio = (relevance_score - 0.45) / 0.4
        blend_ratio = max(0, min(1, blend_ratio))  # Clamp to [0,1]
        
        user_model = adapter_manager.blend_adapters(user_id, model,peft_config, blend_ratio)
        adapter_type = f"blended ({blend_ratio:.2f} local)"
        return user_model, adapter_type
    else:
        # Use discrete selection for clear cases
        user_model = local_model if adapter_type == "local" else global_model
        return user_model, adapter_type


def extract_and_append_features(llm, user_input, graph, retries=3):
    for i in range(retries):
        try:
            preferences = extract_preferences(user_input, llm, extract_preferences_prompt)
            if preferences == {}:
                print("No preferences extracted. ",preferences)
                return  # No preferences extracted, nothing to append
            parsed_preferences = parse_explicit_features(preferences)

            # Don't add questions to the preference graph
            append_features_in_the_graph(graph, parsed_preferences, tokenizer=bert_tokenizer, model=bert_model)
            print("💕 Building graph with preferences")
            return  # Success, exit the function
        except Exception as e:
            print(f"Error extracting and appending features (attempt {i+1}/{retries}): {e}")
            if i == retries - 1:
                print("Failed to extract and append features after multiple attempts.")

# Update process_user_input
def process_user_input(llm, user_input, user_id, graph=None, use_federated=True):
    """
    Process user input and return response, updated graph, and statements.
    
    Args:
        user_input: The user's input text
        user_id: User ID for personalization
        graph: Existing preference graph or None
        use_federated: Whether to use federated learning
    """
    global query_router
    response = None
    adapter_type = "default"  
    statements = []

    # Initialize query router if needed
    if user_id not in user_routers:
        user_routers[user_id] = QueryRouter(embedding_model)
    query_router = user_routers[user_id]    
    
    graph_query = None
    for i in range(3):
        try:
            try:
                print(f"🔄 Attempt {i+1} to classify user input...")
                category_response = query_model(user_input, llm, classification_prompt)

                json_match = re.search(r'(\{.*\})', category_response, re.DOTALL)

                if json_match:
                    json_str = json_match.group(1)
                    try:
                        json_obj = json.loads(json_str)
                        parsed_category_response = parser.parse(json_str)
                    except json.JSONDecodeError as e:
                        print(f"JSON parsing error: {e}")
                        raise ValueError(f"Invalid JSON: {json_str}")
                else:
                    raise ValueError("No JSON found in LLM response")

            except Exception as e:
                print(f"⚠️ Error parsing classification: {e}")
                print(f"Raw response: {category_response}")
                # Fallback to default values
                from types import SimpleNamespace
                parsed_category_response = SimpleNamespace(
                    category="preference",
                    genre_filter=[],
                    preference_type_filter=[]
                )

            parsed_category = apply_stemming(parsed_category_response.category)
            genre_filter = apply_stemming(parsed_category_response.genre_filter)
            preference_type_filter = apply_stemming(parsed_category_response.preference_type_filter)

            parsed_category = parsed_category[0].lower() if parsed_category else "preference"

            print(f"Parsed category: {parsed_category}")
            print(f"Genre filter: {genre_filter}")
            print(f"Preference type filter: {preference_type_filter}")

            is_question = is_input_question(user_input)

            if parsed_category in ["preference", "prefer"] or (parsed_category == "both" and not (is_question or parsed_category in ["query", "queri"])):
                print(f"🔄 Attempt {i+1} to extract preferences...")
                extract_and_append_features(llm, user_input, graph)

            if parsed_category in ["query", "queri"] or parsed_category == "both" or is_question:

                query_text = user_input

                graph_query = query_preferences(
                    graph, 
                    user_id, 
                    query_text, 
                    tokenizer=bert_tokenizer, 
                    model=bert_model, 
                    genre_filter=genre_filter, 
                    preference_type_filter=preference_type_filter, 
                    top_k=5, 
                    active_only=True
                )

                statements = [entry['statement'] for entry in graph_query] if graph_query else []
                print('statements are: ', statements)

                if use_federated:
                    user_model, adapter_type = adapter_decider(user_id, statements, user_input, graph)

                if statements:
                    response = query_huggingface_model(
                        query_text,
                        statements,
                        user_model,  # Use user-specific model
                        tokenizer
                    )
                else:
                    response = query_huggingface_model(
                        query_text,
                        '',
                        user_model,  # Use user-specific model
                        tokenizer
                    )
                response = clean_llm_answer(response)
            return {
                "graph": graph,
                "response": response,
                "adapter_used": adapter_type if use_federated else "default",
                "statements": statements if statements else []
            }
            
        except Exception as e:
            print(f"⚠️ Error processing input: {e}")
            if i < 2:
                print("Retrying...")
                continue
            else:
                print("Failed after 3 attempts.")
                return {
                    "graph": graph,
                    "response": None,
                    "adapter_used": adapter_type if use_federated else "default",
                    "statements": []
                }


def feedback_maker(chats, check):
    rest_of_input = ' '.join(check[1:])
    # Get the last chat entry from the dict
    if chats:
        last_query = list(chats.keys())[-1]
        last_response = chats[last_query]
        return f"Query: {last_query}\nResponse: {last_response}\nFeedback: {rest_of_input}"
    else:
        return f"Query: \nResponse: \nFeedback: {rest_of_input}"

def interactive_cli():
    """Run an interactive command-line interface for preference learning system"""
    
    user_id = "user_2"
    use_federated = True
    chats = {}  # Changed from list to dict

    print("\n===== Preference Learning System =====")
    print("Type 'exit' to quit, 'federate' to perform federated learning")
    print("If Feedback is to be provided type FEEDBACK in capital before prompt, and type FEEDBACK voice for voice input")
    print("=======================================\n")
    
    graph = build_preference_graph(user_id)
    chats = load_chat_history(user_id)
    if not isinstance(chats, dict):
        chats = {}

    while True:
        print("\n> ", end="")
        user_input = input().strip()
        check = user_input.split()
        
        # Handle exit, federate, voice commands as before
        if user_input.lower() == "exit":
            print("Goodbye!")
            break
            
        if user_input.lower() == "federate":
            print("🌐 Starting manual federated learning process...")
            perform_federated_learning(model, peft_config)
            print("✅ Federated learning complete!")
            continue

        if len(check) > 1 and check[0] == "FEEDBACK" and check[1].lower() != "voice" and chats:
            print("Processing text Feedback...")
            structured_feedback = feedback_maker(chats, check)
            extract_and_append_features(llm1, structured_feedback, graph)
            chat_obj = {
                "chat": structured_feedback,
                "voice": None
            }
            run_implicit_feature_extraction(
                chat_obj, llm1, model, sentiment_pipeline,
                embedding_model, whisper_model,
                user_id=user_id,
                adapter_manager=adapter_manager
            )
            continue

        if len(check) > 1 and check[0] == "FEEDBACK" and check[1].lower() == "voice" and chats:
            print("Recording voice Feedback (speak now)...")
            recordings_dir = os.path.join(os.path.dirname(__file__), "recordings")
            os.makedirs(recordings_dir, exist_ok=True)
            
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            recording_filename = f"recording_{timestamp}.wav"
            recording_path = os.path.join(recordings_dir, recording_filename)
            
            user_input = record_and_transcribe_audio(duration=5, output_path=recording_path)
            check = user_input.strip().split()
            
            structured_feedback = feedback_maker(chats, check)
            extract_and_append_features(llm1, structured_feedback, graph)
            chat_obj = {
                "chat": structured_feedback,
                "voice": recording_path
            }
            run_implicit_feature_extraction(
                chat_obj, llm1, model, sentiment_pipeline,
                embedding_model, whisper_model,
                user_id=user_id,
                adapter_manager=adapter_manager
            )
            continue
        
        if user_input:
            # Pass any previous feedback to the processor
            result = process_user_input(
                llm1, user_input, user_id, graph,
                use_federated=use_federated
            )
            
            graph = result["graph"]
            response = result["response"]
            adapter_used = result.get("adapter_used", "default")
            
            if response:
                print(f"\nResponse [{adapter_used}]: {response}")
                chats[user_input] = response  # Store in dict

    # Save final state
    save_chat_history(chats, user_id)
    visualize_interactive_graph(graph)

import gc, torch

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
free_gpu()

def batch_processing(csv_file):
    """Process a CSV file of preferences and questions"""
    output_path = './kaggle/working/preference_evaluation_results.csv'
    first_save = True
    df = pd.read_csv(csv_file)
    print(f"Loaded {len(df)} examples from {csv_file}")

    # Initialize per-user graphs and chat histories
    user_graphs = {}

    # First process preferences for each user
    for idx, row in df.iterrows():
        if idx % 2 == 0:
            llm = llm1
        else:
            llm = llm2
        user_id = row['userID']
        pref = row['preference_prompt']

        if user_id not in user_graphs:
            user_graphs[user_id] = build_preference_graph(user_id)

        print(f"Processing preference for user {user_id}: {pref}")
        extract_and_append_features(llm, pref, user_graphs[user_id])

    total = 0
    correct = 0
    results = []
    llm = llm1

    for idx, row in df.iterrows():
        free_gpu()
        total += 1
        if idx % 2 == 0:
            llm = llm2
        if idx % 2 != 0:
            llm = llm1

        question = row['question_prompt']
        user_id = row['userID']
        print(f"Processing question for user {user_id}: {question}")

        try:
            # Always use the correct user's graph
            graph = user_graphs[user_id]
            result = process_user_input(llm, question, user_id, graph=graph)
            # Update the user's graph in case it changed
            user_graphs[user_id] = result["graph"]
            llm_answer = result["response"]
            statements = result.get("statements", [])

            if not llm_answer:
                # If no answer, skip evaluation and mark as incorrect
                evaluation_output = {"verdict": 0, "feedback": "No response generated."}
            else:
                evaluation_output = evaluate_response(
                    question,
                    row['answer'],
                    row['preference_prompt'],
                    llm_answer,
                    llm=llm
                )
            # evaluation_output is likely a dict, so extract verdict and feedback
            verdict = evaluation_output.get("verdict", "")
            feedback = evaluation_output.get("feedback", "")

            # Store result
            result_data = {
                'userID': row['userID'],
                'preference': row['preference_prompt'],
                'question_prompt': row['question_prompt'],
                'truth_answer': row['answer'],
                'llm_raw_answer': llm_answer,
                'is_correct': verdict,
                'feedback': feedback,
                'statements': statements,
                'category': row['category']
            }
            results.append(result_data)
            print(f"Here is the result data: {result_data}")
            

            # Print progress
            print(f"Example {idx+1}/{len(df)}: {'✓' if verdict == 1 else '✗'} | Expected: {row['answer']}")

            correct += 1 if verdict == 1 else 0

            if feedback.lower() != 'correct' and feedback.lower() != 'correct.':
                structured_feedback = f"Query: {question}\nResponse: {llm_answer}\nFeedback: {feedback}"
                if str(verdict).lower() != 'aligns':
                    extract_and_append_features(llm, structured_feedback, user_graphs[user_id])
                chat_obj = {
                    "chat": structured_feedback,
                    "voice": None
                }
                run_implicit_feature_extraction(
                    chat_obj, llm, model, sentiment_pipeline,
                    embedding_model, whisper_model,
                    user_id=user_id,
                    adapter_manager=adapter_manager
                )
            if idx > 0 and idx % 10 == 0:
                results_df = pd.DataFrame(results)
                write_mode = 'w' if first_save else 'a'
                header = first_save
                results_df.to_csv(output_path, index=False, mode=write_mode, header=header)
                first_save = False
                results = []

                print(f"✅ Saved batch {idx}")
        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            results.append({
                'userID': row['userID'],
                'preference': row['preference_prompt'],
                'question_prompt': row['question_prompt'],
                'truth_answer': row['answer'],
                'llm_raw_answer': str(e),
                'llm_extracted_answer': 'ERROR',
                'is_correct': False,
                'category': row['category']
            })
    if results:  # save any leftover rows
        results_df = pd.DataFrame(results)
        results_df.to_csv(output_path, index=False, mode='a', header=not os.path.exists(output_path))
        results_df = pd.read_csv(output_path)

    
    # Calculate accuracy
    accuracy = correct / total if total > 0 else 0
    print(f"\nFinal Accuracy: {accuracy:.2%} ({correct}/{total})")

    # Create results dataframe and save
    
    results_df = pd.read_csv(output_path)
    
    # Analysis by category
    if 'category' in results_df.columns:
        print("\nAccuracy by category:")
        category_accuracy = results_df.groupby('category')['is_correct'].agg(['mean', 'count']).sort_values('mean', ascending=False)
        print(category_accuracy)

    # Save all user graphs to folder
    for user_id, graph in user_graphs.items():
        visualize_interactive_graph(graph, user_id=user_id)


# 3. Add query_huggingface_model function to main.py
def query_huggingface_model(query_text, preference_statements, model, tokenizer):
    try:
        # build prompt (same as yours)
        if preference_statements:
            preference_text = "\n".join(preference_statements) if isinstance(preference_statements, list) else preference_statements
            prompt = f'''<|system|>You are a helpful assistant. Answer this question '{query_text}'.
            here are the preferences given, if they are not helpful then you can answer the query on your own:\n{preference_text}'''
        else:
            prompt = f"<|system|>You are a helpful assistant.<|user|>{query_text}<|assistant|>"

        # get primary device for model tensors
        model_device = next(model.parameters()).device

        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
        inputs = {k: v.to(model_device) for k, v in inputs.items()}

        generation_kwargs = {
            "max_new_tokens": 40,        # reduce from 70 -> 40
            "do_sample": True,
            "temperature": 0.1,
            "repetition_penalty": 1.1,
            "pad_token_id": tokenizer.pad_token_id,
            "use_cache": True
        }

        import torch
        with torch.inference_mode():
            torch.cuda.empty_cache()    # attempt to free small fragments pre-gen
            output = model.generate(**inputs, **generation_kwargs)

        response = tokenizer.decode(output[0], skip_special_tokens=True)
        if "<|assistant|>" in response:
            response = response.split("<|assistant|>")[-1].strip()
        return response
    except Exception as e:
        print(f"Error in query_huggingface_model: {e}")
        return "I'm sorry, I encountered an error while trying to answer."



# Runner

In [ ]:
if __name__ == "__main__":
    # Emulate argparse in Kaggle
    class Args:
        batch = "/kaggle/input/pref-data/preference_cleaned.csv"
        federate = False
        no_federated = False

    args = Args()

    # Handle federated learning command
    if args.federate:
        print("Running federated learning aggregation...")
        global_model = perform_federated_learning(model, peft_config)
    else:
        use_federated = not args.no_federated
        batch_processing(args.batch)